In [ ]:
from pathlib import Path
from functools import reduce


def read_csv(spark, path):
    return (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .csv(path)
    )


def validate_schema(df, expected_columns):
    incoming_columns = set(df.columns)
    expected_columns = set(expected_columns)

    missing_columns = expected_columns - incoming_columns
    new_columns = incoming_columns - expected_columns

    if missing_columns:
        return "REJECT", missing_columns, new_columns

    if new_columns:
        return "EVOLVED", missing_columns, new_columns

    return "VALID", missing_columns, new_columns


def read_csv_dataset(
    spark,
    path,
    expected_columns
):
    files = list(Path(path).glob("*.csv"))

    valid_dataframes = []

    for file in files:
        df = read_csv(
            spark,
            str(file)
        )

        status, missing_columns, new_columns = validate_schema(
            df,
            expected_columns
        )

        if status == "REJECT":
            print(
                f"Rejected {file}. "
                f"Missing columns: {missing_columns}"
            )

            (
                df.write
                .mode("append")
                .parquet(
                    "data/quarantine/communications/"
                )
            )

            continue

        if status == "EVOLVED":
            print(
                f"Schema evolution detected in {file}. "
                f"New columns: {new_columns}"
            )

        valid_dataframes.append(df)

    return reduce(
        lambda left, right:
            left.unionByName(
                right,
                allowMissingColumns=True
            ),
        valid_dataframes
    )


def read_json_dataset(
    spark,
    path,
    expected_columns
):
    df = spark.read.json(path)

    validate_schema(
        df,
        expected_columns
    )

    return df


def read_txt_dataset(
    spark,
    path,
    expected_columns
):
    df = spark.read.text(path)

    return df


readers = {
    "csv": read_csv_dataset,
    "json": read_json_dataset,
    "txt": read_txt_dataset
}


for dataset_name, config in datasets.items():

    path = config["path"]

    file_format = config["format"]

    expected_columns = config.get(
        "expected_columns"
    )

    reader = readers.get(file_format)

    if reader is None:
        raise ValueError(
            f"Unsupported format: {file_format}"
        )

    df = reader(
        spark,
        path,
        expected_columns
    )